# 🧠 Lesson 01: Core Concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HosseinHeydari2004/RAG-Tutorial-From-Zero-to-Advanced/blob/main/notebooks/01_core_concepts.ipynb)

> **Why do we need RAG at all? Let's break a language model first, then fix it.**

---

## 🎯 Goal
By the end of this lesson you will understand the three problems RAG solves, and how it compares to the other options.

## ℹ️ Lesson info

| | |
|---|---|
| ⏱ **Time** | ~15 minutes |
| 📚 **Prerequisites** | [Lesson 00](00_rag_in_20_lines.ipynb) |
| 🔑 **API key** | Required (OpenRouter) |
| 🛠 **Libraries** | `langchain`, `langchain-openrouter` |

In [ ]:
!pip install -q -U "langchain>=1" langchain-openrouter

In [1]:
from getpass import getpass
import os
from langchain.chat_models import init_chat_model

In [2]:
# set API Key
if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

In [6]:
model = init_chat_model(
    model="openai/gpt-4o-mini",
    model_provider="openrouter"
)

respone = model.invoke(
    input=[
        {
            "role":"user",
            "content":"How many days of paid time off (PTO) is an employee with 2 years of experience at Vexonova Holding entitled to?"
        }
    ]
)

print(respone.content)

I'm sorry, but I don't have specific information about the PTO policies of Vexonova Holding or any other specific company. PTO policies can vary widely by employer, including factors like the number of years an employee has been with the company, job position, and the company's overall policy. It's best to refer to the company's employee handbook or HR department for the most accurate information regarding PTO entitlement.


## ❌ Problem 1: The model doesn't know your data

Look at the answer above. A good model is honest: it says it has no information about this company. It *can't* know. Your company handbook, your emails, your database were never part of its training data.

No matter how smart the model is, it cannot answer questions about documents it has never seen.

In [7]:
questions = [
    "How many days of paid time off (PTO) is an employee with 2 years of experience at Vexonova Holding entitled to? Answer with a single number only.",
    "What is the exact remote work policy at Vexonova Holding? Answer in one sentence.",
]

for question in questions:
    respone = model.invoke(
        input=[
            {
                "role": "user",
                "content": question
            }
        ]
    )
    print("Q:", question)
    print("A:", respone.content)
    print()

Q: How many days of paid time off (PTO) is an employee with 2 years of experience at Vexonova Holding entitled to? Answer with a single number only.
A: 20

Q: What is the exact remote work policy at Vexonova Holding? Answer in one sentence.
A: I don't have access to specific company policies for Vexonova Holding. For the exact remote work policy, please check their official website or contact their HR department.



## 🎭 Problem 2: The model makes things up (hallucination)

In our run, the first question got the answer **20**, said with full confidence. But the model has never seen this company's handbook. The number is a *guess* that sounds like a fact. (In Lesson 00, the handbook said **15** for two years.)

The second question shows the other side: when the model was free to answer openly, it admitted it didn't know.

This is called **hallucination**: a fluent, confident answer that is not grounded in any real source. It's dangerous because it is hard to spot.

Let's check how stable that guess is. We ask the *same* question three times:

In [8]:
question = "How many days of paid time off (PTO) is an employee with 2 years of experience at Vexonova Holding entitled to? Answer with a single number only."

for i in range(3):
    respone = model.invoke(
        input=[
            {
                "role": "user",
                "content": question
            }
        ]
    )
    print(f"Run {i + 1}:", respone.content)

Run 1: 15
Run 2: 15
Run 3: 10


### 👀 What we saw

Same question, three runs, and the answers were **15, 15, 10**. If the model really knew the policy, it would not change its answer. The number is a guess.

The scary part: one guess may even *look* right. But there is no way to tell from the outside. A hallucination sounds exactly like a fact.

## 🕰️ Problem 3: The model's knowledge is frozen

A language model learns from text collected up to a certain date, its *training cutoff*. After that, it learns nothing new. Prices change, laws change, your company updates its handbook, but the model still "remembers" the old world.

Retraining a model to fix this is slow and very expensive. We need a cheaper way to give the model fresh information.

## 📋 The three problems

| # | Problem | Example |
|---|---|---|
| 1 | **Private data**: it never saw your documents | "What is *our* PTO policy?" |
| 2 | **Hallucination**: it guesses when it doesn't know | 15, 15, 10 for the same question |
| 3 | **Frozen knowledge**: it can't know what happened after training | "What changed in the new handbook?" |

All three have the same root cause: **the model doesn't have the right information at the moment it answers.** So what if we simply *hand it the right information* together with the question?

## 🪟 The context window

A model can only read a limited amount of text at once. This limit is called the **context window**. Everything must fit inside it: your question, any documents you give it, and the answer it writes.

The size is measured in **tokens** (small pieces of words, roughly a few characters each) and it differs from model to model. Check your model's page on OpenRouter for its exact limit.

So the simplest idea is: *paste the whole document into the prompt.* Let's try it with the handbook from Lesson 00:

In [9]:
text = """Paid Time Off (PTO)
Employees with 0 to 1 years of service get 10 PTO days per year. Employees with 1 to 3 years get 15 days. Employees with more than 3 years get 20 days.

Remote Work
Employees may work remotely up to 3 days per week. Remote days must be agreed with the team lead in advance.

Equipment
The company provides one laptop per employee. Broken or lost equipment must be reported to IT within 24 hours.

Expense Reimbursement
Travel and work-related expenses are reimbursed within 30 days. Receipts must be submitted through the finance portal."""

question = "How many vacation days do I get after two years?"

respone = model.invoke(
    input=[
        {
            "role": "system",
            "content": "Answer the question using only the context. If the answer is not in the context, say \"I don't know\"."
        },
        {
            "role": "user",
            "content": f"Context:\n{text}\n\nQuestion: {question}"
        }
    ]
)
print(respone.content)
print("cost ($):", respone.response_metadata.get("cost"))

You get 15 vacation days after two years.
cost ($): 3.255e-05


### 🧪 What if the document is bigger?

Our handbook has 4 short policies. A real company handbook has hundreds of pages. To simulate this, we repeat the same text 50 times and ask the same question. Watch the cost.

In [10]:
big_text = (text + "\n\n") * 50

respone = model.invoke(
    input=[
        {
            "role": "system",
            "content": "Answer the question using only the context. If the answer is not in the context, say \"I don't know\"."
        },
        {
            "role": "user",
            "content": f"Context:\n{big_text}\n\nQuestion: {question}"
        }
    ]
)
print(respone.content)
print("characters sent:", len(big_text))
print("cost ($):", respone.response_metadata.get("cost"))

You get 15 vacation days after two years.
characters sent: 27950
cost ($): 0.00097335


### 👀 What we saw

| | Characters sent | Cost per question |
|---|---|---|
| Original handbook (4 policies) | ~560 | ~$0.00003 |
| Handbook repeated 50 times | 27,950 | ~$0.00097 |

The answer stayed correct, but the cost grew about **30×**. And this "big" document is still tiny: a real handbook, or a whole company's files, is far larger.

Pasting everything into the prompt means you **pay for all of it on every single question**, even though the answer lives in just one small chunk. It is also limited by the context window, and models tend to work better when they only see what's relevant.

> ℹ️ Numbers are from one run with one model at today's prices. Yours will differ a bit.

## ⚖️ Three ways to give a model your knowledge

| | **Long prompt** | **Fine-tuning** | **RAG** |
|---|---|---|---|
| **Idea** | Paste all documents into the prompt | Train the model further on your data | Find the relevant chunks, paste only those |
| **Cost per question** | High (grows with document size) | Low at question time, but training costs upfront | Low (small prompt) |
| **Document size limit** | The context window | Needs lots of examples and effort | Practically no limit on the *library*; only the chunks must fit |
| **Updating knowledge** | Edit the text | Retrain | Re-index only what changed |
| **Can show its source?** | Not easily | Hard | Yes, the retrieved chunk is the source |
| **Best for** | Tiny documents | Teaching a *style, format or behavior* | Answering from *large, changing documents* |

In practice, many real systems combine them. For "answer questions from our documents", RAG is usually the first thing to try.

## 🤔 When is RAG *not* the right tool?

RAG is not magic. Skip it (or think twice) when:

- **Your document is tiny.** If it fits comfortably in the prompt, as our 4-policy handbook did, just paste it. Simpler is better.
- **You want to change the model's behavior or style**, such as a tone of voice or a strict output format. That is closer to fine-tuning or better prompting.
- **The question needs the whole document at once**, such as "summarize this entire report". RAG retrieves *pieces*, so it can miss the big picture.
- **The answer isn't written anywhere.** RAG can only find what exists in your documents. If the knowledge is missing, retrieval returns something irrelevant.

A good habit: start with the simplest approach that works, and add RAG when the simple one starts to hurt.

## 📌 Summary

- A language model doesn't know your **private data**, it can **hallucinate** (we saw 15, 15, 10 for the same question), and its knowledge is **frozen** at its training date.
- All three share one root cause: the model lacks the right information *at the moment it answers*.
- The **context window** limits how much text a model can read at once. Pasting everything into the prompt works for tiny documents, but its cost grew about **30×** in our test.
- **RAG** fixes this by retrieving only the relevant chunks and handing those to the model.
- RAG is not always the answer. For tiny documents, or when you want to change *behavior*, other tools fit better.

## ✍️ Exercise (10 minutes)

1. In the "bigger document" cell, change `50` to `10`, then to `100`. Record the cost each time.
2. Is the growth in cost roughly proportional to the number of characters?

**Success:** you can explain in one sentence why a long prompt gets expensive.

> ⚠️ If you hit a length error with `100`, that is the context window limit in action. Use a smaller number.

## 🔥 Challenge (optional)

Find a question about a real-world fact where the model gives a **confident but wrong** answer (hint: try very specific or obscure details). Run it three times. Do the answers change?

Then ask yourself: how would you *prove* the answer is wrong, if you didn't already know the truth? This is exactly why we will learn to **evaluate** RAG systems later in the course.

## 📖 Next

Now we know *why* RAG exists. Time to build each piece properly, starting with the first one: getting documents into Python.

**➡️ Next lesson:** [02 Loading](02_loading.ipynb)